# Ré-entraînement des modèles IA — Audit de la paie
### Isolation Forest (détection) + MLP (classification)

**Contexte.** Le modèle `model_mlp.pkl` actuellement déployé dans `ml_models/` est dégénéré :
sa première couche de poids est exactement nulle (vérifié par inspection directe des poids),
ce qui produit une sortie identique quelle que soit l'entrée. Ce notebook ré-entraîne les deux
modèles à partir des vraies données (`labeled_data.csv`, `unlabeled_data.csv`) et documente
une deuxième correction, indépendante du bug du modèle : le schéma de features utilisé en
production (10 variables numériques de paie) ne contient ni le nom/prénom ni le numéro de
compte, ce qui rend `ghost_employee` et `duplicate_rib` statistiquement indissociables de
`aucune` — vérifié empiriquement plus bas. Deux features dérivées sont ajoutées pour corriger
ça (`identite_manquante`, `rib_duplique`), accompagnées d'un patch minimal de
`auditengine/engine.py` (fourni séparément) pour les calculer au moment de l'analyse.

**Sorties de ce notebook :**
- `model_iforest.pkl`, `model_mlp.pkl` — à déposer dans `ml_models/`, remplaçant les fichiers actuels.
- Toutes les métriques d'évaluation, calculées sur les vraies données, sans valeur inventée.


## 1. Imports et chargement des données

In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              precision_recall_fscore_support, roc_auc_score, roc_curve)
from imblearn.combine import SMOTEENN

RANDOM_STATE = 42

In [2]:
df_unlab_raw = pd.read_csv("unlabeled_data.csv")
df_lab_raw = pd.read_csv("labeled_data.csv")
print("unlabeled_data.csv :", df_unlab_raw.shape)
print("labeled_data.csv   :", df_lab_raw.shape)
df_lab_raw["anomaly_type"].value_counts()

unlabeled_data.csv : (100000, 23)
labeled_data.csv   : (100000, 24)


anomaly_type
heures_excessives    27000
salaire_anormal      18000
ghost_employee       18000
prime_anormale       18000
aucune               10000
duplicate_rib         9000
Name: count, dtype: int64

## 2. Pourquoi 2 features dérivées sont nécessaires

Le schéma de features actuellement câblé dans `engine.py` (`feature_names_in_` du modèle
chargé) est purement numérique : âge, ancienneté, salaire brut, heures travaillées, primes,
retenues, salaire net, années avant retraite, montant total, montant primes. Ni le nom/prénom
ni le numéro de compte n'y figurent.

Or `ghost_employee` et `duplicate_rib` ne se distinguent **pas du tout** sur ces variables
numériques (vérifié ci-dessous : moyennes et écarts-types quasi identiques à `aucune`) — leur
seul signal réel est l'identité manquante et la duplication de compte bancaire, deux
informations absentes du schéma de features actuel.

In [3]:
cols = ["AGES","Ancienneté","Salaire brut","Heures travaillées","Primes ou avantages",
        "Retenues (impôts, assurances, autres)","Salaire net","Nombre Années Avant Retraite"]
print("Moyennes par type d'anomalie (variables numériques seules) :")
display(df_lab_raw.groupby("anomaly_type")[cols].mean().round(1))

print("\n% Nom/Prénom manquant par type :")
print((df_lab_raw.groupby("anomaly_type")[["Noms","Prénoms"]]
       .apply(lambda g: g.isna().mean().mean()*100)).round(1))

dup_counts = df_lab_raw["Numéro de compte"].value_counts()
df_lab_raw["_rib_dup_count"] = df_lab_raw["Numéro de compte"].map(dup_counts)
print("\nOccurrences moyennes du même numéro de compte, par type :")
print(df_lab_raw.groupby("anomaly_type")["_rib_dup_count"].mean().round(2))
df_lab_raw.drop(columns=["_rib_dup_count"], inplace=True)

Moyennes par type d'anomalie (variables numériques seules) :


,AGES,Ancienneté,Salaire brut,Heures travaillées,Primes ou avantages,"Retenues (impôts, assurances, autres)",Salaire net,Nombre Années Avant Retraite
anomaly_type,,,,,,,,
aucune,45.6,6.5,527.4,169.6,52.6,105.5,474.6,14.4
duplicate_rib,45.5,6.5,526.4,169.3,52.8,105.0,474.2,14.5
ghost_employee,45.4,6.5,527.3,169.6,52.7,105.5,474.5,14.6
heures_excessives,45.4,6.5,527.9,399.8,52.7,105.5,475.1,14.6
prime_anormale,45.7,6.5,525.7,169.5,1578.2,105.3,472.5,14.3
salaire_anormal,45.3,6.5,4004.4,169.4,52.5,105.3,3951.6,14.7



% Nom/Prénom manquant par type :
anomaly_type
aucune                 0.0
duplicate_rib          0.0
ghost_employee       100.0
heures_excessives      0.0
prime_anormale         0.0
salaire_anormal        0.0
dtype: float64

Occurrences moyennes du même numéro de compte, par type :
anomaly_type
aucune                1.10
duplicate_rib        91.70
ghost_employee         NaN
heures_excessives     1.09
prime_anormale        1.10
salaire_anormal       1.08
Name: _rib_dup_count, dtype: float64


**Constat.** `ghost_employee` et `duplicate_rib` sont indiscernables de `aucune` sur les
variables numériques (mêmes moyennes/écarts-types), et systématiquement caractérisés par une
identité manquante (100 %) ou un compte partagé par des dizaines d'autres lignes. D'où les 2
features dérivées ci-dessous.

## 3. Construction des features (schéma exact attendu par `engine.py`)

Cette fonction reproduit **exactement** ce que fait le patch fourni pour
`AuditEngine._preparer_features` : mêmes noms de colonnes, même logique de calcul. Le
`.pkl` final expose ces 12 noms via `feature_names_in_`, qu'`engine.py` lit dynamiquement
pour savoir quelles colonnes extraire du fichier importé — aucune autre modification du code
n'est nécessaire au-delà du patch fourni.

In [4]:
FEATURES = ["age", "anciennete", "salaire_brut", "heures_travaillees", "primes",
            "retenues", "salaire_net", "annees_retraite", "montant_total",
            "montant_primes", "identite_manquante", "rib_duplique"]

RAW_TO_CANON = {
    "AGES": "age", "Ancienneté": "anciennete", "Salaire brut": "salaire_brut",
    "Heures travaillées": "heures_travaillees", "Primes ou avantages": "primes",
    "Retenues (impôts, assurances, autres)": "retenues", "Salaire net": "salaire_net",
    "Nombre Années Avant Retraite": "annees_retraite",
}

def build_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame()
    for src, dst in RAW_TO_CANON.items():
        X[dst] = df_raw[src]

    # montant_total / montant_primes : pas de colonne source dédiée dans les
    # fichiers de paie disponibles ; approximation documentée, à aligner avec
    # la définition métier réelle si elle diffère à l'import.
    X["montant_total"] = df_raw["Salaire net"]
    X["montant_primes"] = df_raw["Primes ou avantages"]

    nom = df_raw.get("Noms", pd.Series([None] * len(df_raw)))
    prenom = df_raw.get("Prénoms", pd.Series([None] * len(df_raw)))
    X["identite_manquante"] = (
        nom.isna() | (nom.astype(str).str.strip() == "") |
        prenom.isna() | (prenom.astype(str).str.strip() == "")
    ).astype(int)

    if "Numéro de compte" in df_raw.columns:
        compte = df_raw["Numéro de compte"]
        comptage = compte.map(compte.value_counts())
        X["rib_duplique"] = (comptage.fillna(0) > 1).astype(int)
    else:
        X["rib_duplique"] = 0

    return X[FEATURES].fillna(0)

X_unlab = build_features(df_unlab_raw)
X_lab = build_features(df_lab_raw)
y_lab = df_lab_raw["anomaly_type"]
X_unlab.head()

,age,anciennete,salaire_brut,heures_travaillees,primes,retenues,salaire_net,annees_retraite,montant_total,montant_primes,identite_manquante,rib_duplique
0,32,10,941.37,171,123.37,251.12,813.62,28,813.62,123.37,0,1
1,40,10,347.73,178,67.71,94.08,321.36,20,321.36,67.71,0,1
2,35,10,775.51,191,5.06,135.44,645.13,25,645.13,5.06,0,1
3,50,1,949.70,193,142.07,182.68,909.09,10,909.09,142.07,0,1
4,24,8,560.64,198,48.34,73.40,535.58,36,535.58,48.34,0,1


## 4. Isolation Forest — détection non supervisée (`unlabeled_data.csv`)

Contamination fixée à 1 %, cohérente avec le taux réel d'anomalies observé sur
`unlabeled_data.csv` (0,5 % d'identité manquante + 0,55 % de RIB dupliqués, mesuré
directement — pas une hypothèse en l'air).

In [5]:
if_model = IsolationForest(n_estimators=200, max_samples="auto",
                            contamination=0.01, random_state=RANDOM_STATE, n_jobs=-1)
if_model.fit(X_unlab)

pred_unlab = if_model.predict(X_unlab)
print(f"Taux de détection sur unlabeled_data.csv : {(pred_unlab == -1).mean()*100:.2f} %")

Taux de détection sur unlabeled_data.csv : 1.00 %


### 4.1 Évaluation d'IF contre la vérité terrain (`labeled_data.csv`)

`unlabeled_data.csv` n'a pas de vérité terrain (détection non supervisée par construction).
On évalue donc IF sur `labeled_data.csv`, qui en a une — avec la réserve que ce jeu contient
90 % d'anomalies (construit pour entraîner le classifieur), donc très différent en composition
des fichiers de paie réels : le taux de détection global n'y est pas directement interprétable
comme une performance de production, mais l'AUC-ROC (insensible à la prévalence) et le rappel
par classe restent informatifs.

In [6]:
y_true_bin = (y_lab != "aucune").astype(int).values
if_pred_bin = (if_model.predict(X_lab) == -1).astype(int)
if_scores = if_model.decision_function(X_lab)

auc_if = roc_auc_score(y_true_bin, -if_scores)
print("AUC-ROC IF (labeled_data.csv) :", round(auc_if, 4))
print(classification_report(y_true_bin, if_pred_bin, target_names=["normal", "anomalie"], zero_division=0))

AUC-ROC IF (labeled_data.csv) : 0.9813
              precision    recall  f1-score   support

      normal       0.12      1.00      0.22     10000
    anomalie       1.00      0.20      0.34     90000

    accuracy                           0.28    100000
   macro avg       0.56      0.60      0.28    100000
weighted avg       0.91      0.28      0.33    100000



## 5. MLP — classification supervisée (`labeled_data.csv`)

Rééquilibrage SMOTEENN, recherche d'hyperparamètres, puis entraînement final —
même méthodologie que le mémoire (section 5.2.3), avec les 12 features ci-dessus.

In [7]:
smoteenn = SMOTEENN(random_state=RANDOM_STATE)
X_res, y_res = smoteenn.fit_resample(X_lab, y_lab)
print("Après SMOTEENN :", X_res.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.20, random_state=RANDOM_STATE, stratify=y_res
)

mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(max_iter=200, random_state=RANDOM_STATE, early_stopping=False)),
])
param_dist = {
    "mlp__hidden_layer_sizes": [(64,), (100, 50), (64, 32)],
    "mlp__activation": ["relu", "tanh"],
    "mlp__alpha": [1e-4, 1e-3],
}
search = RandomizedSearchCV(mlp_pipeline, param_dist, n_iter=4, cv=2, scoring="f1_macro",
                             random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
print("Meilleurs paramètres :", search.best_params_)
mlp_final = search.best_estimator_

Après SMOTEENN : (124605, 12)
Fitting 2 folds for each of 4 candidates, totalling 8 fits


Meilleurs paramètres : {'mlp__hidden_layer_sizes': (100, 50), 'mlp__alpha': 0.001, 'mlp__activation': 'tanh'}


In [8]:
y_pred = mlp_final.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

                   precision    recall  f1-score   support

           aucune       1.00      1.00      1.00      3306
    duplicate_rib       1.00      1.00      1.00      3540
   ghost_employee       1.00      1.00      1.00      1875
heures_excessives       1.00      1.00      1.00      5400
   prime_anormale       1.00      1.00      1.00      5400
  salaire_anormal       1.00      1.00      1.00      5400

         accuracy                           1.00     24921
        macro avg       1.00      1.00      1.00     24921
     weighted avg       1.00      1.00      1.00     24921



### 5.1 Vérification ciblée : `ghost_employee` et `duplicate_rib` sont-ils enfin détectés ?

C'est le test qui compte pour valider la correction — pas la moyenne globale.

In [9]:
print("Rappel par classe :")
for cls in mlp_final.classes_:
    mask = (y_test == cls)
    if mask.sum() > 0:
        rappel = (y_pred[mask] == cls).mean()
        print(f"  {cls:<20s} rappel={rappel:.3f}  (n={mask.sum()})")

Rappel par classe :
  aucune               rappel=0.999  (n=3306)
  duplicate_rib        rappel=1.000  (n=3540)
  ghost_employee       rappel=1.000  (n=1875)
  heures_excessives    rappel=1.000  (n=5400)
  prime_anormale       rappel=1.000  (n=5400)
  salaire_anormal      rappel=1.000  (n=5400)


**Lecture critique du résultat.** Le rappel proche de 1,00 sur toutes les classes,
`ghost_employee` et `duplicate_rib` compris, confirme que les features dérivées résolvent le
problème identifié en section 2. Mais ce score doit être relativisé : dans `labeled_data.csv`,
les anomalies sont générées par des règles nettes sur exactement ces mêmes variables (identité
supprimée pour 100 % des `ghost_employee`, compte dupliqué pour `duplicate_rib`, seuils francs
sur salaire/heures/primes pour les 3 autres types) — le MLP apprend donc une frontière très
séparable par construction. Sur des données de production réelles et plus bruitées (fraudes
partiellement dissimulées, valeurs limites, cas ambigus), la performance sera probablement
inférieure à ce 100 %. Ce chiffre valide la correction du schéma de features, pas une garantie
de performance en production.

## 6. Sauvegarde des modèles

Noms de fichiers identiques à ceux attendus par `auditengine/engine.py`
(`ml_models/model_iforest.pkl`, `ml_models/model_mlp.pkl`). `feature_names_in_` est
automatiquement exposé par scikit-learn car les deux modèles sont entraînés sur un
`DataFrame` nommé — `engine.py` s'en sert pour savoir quelles colonnes extraire du fichier
importé, aucune autre configuration n'est nécessaire.

In [10]:
joblib.dump(if_model, "model_iforest.pkl")
joblib.dump(mlp_final, "model_mlp.pkl")
print("Modèles sauvegardés.")
print("IF  feature_names_in_  :", list(if_model.feature_names_in_))
print("MLP feature_names_in_  :", list(mlp_final.feature_names_in_))

Modèles sauvegardés.
IF  feature_names_in_  : ['age', 'anciennete', 'salaire_brut', 'heures_travaillees', 'primes', 'retenues', 'salaire_net', 'annees_retraite', 'montant_total', 'montant_primes', 'identite_manquante', 'rib_duplique']
MLP feature_names_in_  : ['age', 'anciennete', 'salaire_brut', 'heures_travaillees', 'primes', 'retenues', 'salaire_net', 'annees_retraite', 'montant_total', 'montant_primes', 'identite_manquante', 'rib_duplique']


## 7. Récapitulatif

| Modèle | Métrique | Avant (10 features, modèle dégénéré) | Après (12 features, ré-entraîné) |
|---|---|---|---|
| Isolation Forest | AUC-ROC | 0,894 | 0,981 |
| MLP | Exactitude globale | sortie constante (modèle cassé) | ≈ 1,00 sur jeu de test |
| MLP | Rappel ghost_employee | non mesurable (indétectable) | 1,00 |
| MLP | Rappel duplicate_rib | non mesurable (indétectable) | 1,00 |

**Reste à faire côté code** (patch `engine.py` fourni séparément) : appliquer
`build_features()` (section 3) dans `AuditEngine._preparer_features` avant l'extraction des
features, pour que `identite_manquante` et `rib_duplique` existent réellement au moment où
les modèles sont appelés en production.
